## 1. Install Necessary Libraries

This section installs all the required Python libraries for this task. These libraries are essential for tasks such as:
- **Model Handling & Training:** `transformers`, `datasets`, `accelerate`, `peft`, `trl`, `bitsandbytes`, `torch` for loading, fine-tuning, and managing models.
- **Data Processing & Web Scraping:** `beautifulsoup4`, `lxml` for parsing HTML  content of the dataset.
- **Data Analysis & Visualization:** `pandas` for data manipulation, `matplotlib` and `seaborn` for creating plots and visualizations.

In [ ]:

import sys
!{sys.executable} -m pip install -r requirements.txt

## 2. Load and Configure the Pre-trained Model with QLoRA and LoRA

This section focuses on loading the pre-trained language model (`Qwen/Qwen2-1.5B-Instruct`) and preparing it for fine-tuning using QLoRA and LoRA.

**Key steps include:**

- **Model ID Definition:** Specifies the `Qwen/Qwen2-1.5B-Instruct` model from Hugging Face.
- **QLoRA Configuration (`BitsAndBytesConfig`):**
    - Enables 4-bit loading (`load_in_4bit=True`).
    - Sets the 4-bit quantization type to `nf4`.
    - Specifies `torch.bfloat16` as the computation data type for 4-bit layers.
    - Enables double quantization (`bnb_4bit_use_double_quant=True`) for further memory savings.
- **LoRA Configuration (`LoraConfig`):**
    - Sets the LoRA rank (`r=16`) and alpha (`lora_alpha=32`).
    - Applies dropout (`lora_dropout=0.1`) to LoRA layers for regularization.
    - Specifies that biases are not trained (`bias="none"`).
    - Sets the task type to causal language modeling (`task_type="CAUSAL_LM"`).
    - Targets specific modules (`q_proj`, `k_proj`, `v_proj`, `o_proj`) for LoRA adaptation. These are the attention mechanism projections.
- **Applying LoRA:** The `get_peft_model` function applies the LoRA configuration to the base model, creating `lora_model`.
- **Verification:** The number of trainable parameters is printed to confirm that only the LoRA layers (and not the entire model) are set to be trained, significantly reducing computational overhead.

In [ ]:


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Define the model ID for Qwen2 1.5B
model_id = "Qwen/Qwen2-1.5B-Instruct"

# 2. Configure QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Load the model with quantization QLoRA
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)


if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id


model = prepare_model_for_kbit_training(model)
model.train()

# 5. Define LoRA configuration

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,       # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

    ],
)


lora_model = get_peft_model(model, lora_config)

# 6. Print trainable parameters to verify
lora_model.print_trainable_parameters()

print(f"Model '{model_id}' loaded successfully with QLoRA configuration.")
print(f"Tokenizer pad token: {tokenizer.pad_token}, pad_token_id: {tokenizer.pad_token_id}")
print(f"Model pad_token_id: {model.config.pad_token_id}")

## 3. Load and Prepare the Preference Dataset

This section handles the loading and preparation of the preference dataset for training the DPO model. The dataset is  in a JSON format and will be transformed into a structure suitable for the `trl` library.

**Key steps involved:**
- **Load Raw JSON Data:**
    - The script loads data from `preference_dataset.json`. Therefore please ensure that you have the data file named like this and added to the runtime or you can also mount it.
    - It includes error handling for `FileNotFoundError` (if the file doesn't exist) and `json.JSONDecodeError` (if the file is not valid JSON).
- **Process and Transform Data:**
    
    - The goal is to create entries with `prompt`, `chosen` (the preferred completion), and `rejected` (the less preferred completion) fields.
    - Entries with unexpected style field values are skipped with a warning.

- **Convert to Hugging Face Dataset:**
    - The list of processed dictionaries is converted into a Hugging Face `Dataset` object. This format is optimized for use with Hugging Face `transformers` and `trl`.
- **Create Train/Validation Splits:**
    - The `Dataset` is split into training and validation sets (80% for training, 20% for validation).
    - The splits are stored in a `DatasetDict`.




**Note**

I added the function (`preprocess_html`) for processing the HTML content of the completions that can potentially enhance the DPO training process. The function is not executed in the notebook due to the free GPU runtime ending (I decided to see how the fine tuned model perform without changing HTML content at first to see if this is needed at all).

I thought this preprocessing of HTML can be beneficial because it will allow the model to focus more on the slide design (CSS styles, colors, fonts...) and allow the model to concentrate less on the slide and text content. This makes stylistic differences more explicit, which is crucial for the preference learning task.

Therfore the preprocessing includes  

   - Parseing HTML content to identify and replace specific content (like title,name) with  placeholders
   - Preserving structural meaning while abstracting away specific text content
   - Converting CSS style formats to standardized representations
   - Normalizeing hex colors (e.g., `#FFF` → `#FFFFFF`)
   - Standardizing color formats (RGB → hex)


  
> If you wish to explore the potential benefits of HTML preprocessing, you can easily integrate it by calling the `preprocess_html` function on the `chosen` and `rejected` fields before constructing the `data_dict` for training.

In [ ]:
import json
from datasets import Dataset, DatasetDict
import re
from bs4 import BeautifulSoup

def normalize_style(css_value):
    """
    Normalize CSS style values for consistency.


    """
    # Convert short hex colors to full hex

    if re.match(r'^#[0-9a-fA-F]{3}$', css_value):
        r, g, b = css_value[1], css_value[2], css_value[3]
        return f'#{r}{r}{g}{g}{b}{b}'

    # Convert rgb to hex
    rgb_match = re.match(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', css_value)
    if rgb_match:
        r, g, b = map(int, rgb_match.groups())
        return f'#{r:02x}{g:02x}{b:02x}'

    # Normalize font sizes (could add more normalizations)
    if css_value.endswith('px'):
        # Round pixel values
        try:
            px_value = float(css_value[:-2])
            return f"{round(px_value)}px"
        except ValueError:
            pass

    return css_value

def preprocess_html(html_content, prompt=None):
    """
    Preprocess HTML content by replacing specific content (like title or topic) with placeholders
    and normalizing style attributes.


    """
    # Parse HTML
    soup = BeautifulSoup(html_content, 'lxml')

    # Extract key entities from prompt if available
    key_entities = {}
    if prompt:
        # Extract title
        title_match = re.search(r'(?:about|for|on)\s+["\']?([^"\',.]+)["\']?', prompt, re.IGNORECASE)
        if title_match:
            key_entities['TITLE'] = title_match.group(1).strip()

        # Extract name
        name_match = re.search(r'(?:for|by)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2})', prompt)
        if name_match:
            key_entities['NAME'] = name_match.group(1).strip()

        # Extract date if present
        date_match = re.search(r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\w+ \d{1,2},? \d{4})', prompt)
        if date_match:
            key_entities['DATE'] = date_match.group(1).strip()

    # replace content with placeholders
    for i, heading in enumerate(soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])):
        # Save original text
        original_text = heading.get_text().strip()

        # Check if heading contains any key entity and replace
        replaced = False
        for entity_name, entity_value in key_entities.items():
            if entity_value in original_text:
                heading.string = original_text.replace(entity_value, f"{{{entity_name}}}")
                replaced = True
                break


    # Normalize style attributes
    for element in soup.find_all(style=True):
        style_text = element['style']
        style_pairs = re.findall(r'([^:;]+):\s*([^;]+)', style_text)

        normalized_pairs = []
        for prop, value in style_pairs:
            prop = prop.strip()
            value = value.strip()
            normalized_value = normalize_style(value)
            normalized_pairs.append(f"{prop}: {normalized_value}")

        element['style'] = '; '.join(normalized_pairs)

    # Process link text
    for link in soup.find_all('a'):
        if link.string and len(link.string) > 10:
            link.string = "{LINK_TEXT}"

    # Return the preprocessed HTML
    return str(soup)






# 1. Load the JSON dataset
file_path = 'preference_dataset.json'
try:
    with open(file_path, 'r') as f:
        raw_dataset = json.load(f)
    print(f"Successfully loaded {len(raw_dataset)} entries from {file_path}")
except FileNotFoundError:
    print(f"ERROR: The file {file_path} was not found.")
    raw_dataset = []
except json.JSONDecodeError:
    print(f"ERROR: The file {file_path} is not a valid JSON file.")
    raw_dataset = []


# 2. Transform the data into the required format: {"prompt": ..., "chosen": ..., "rejected": ...}
processed_data = []
for entry in raw_dataset:

    if entry.get("style_b") == "chosen" and entry.get("style_a") == "rejected":
        processed_data.append({
            "prompt": entry["prompt"],
            "chosen": entry["completion_b"],
            "rejected": entry["completion_a"]
        })
    elif entry.get("style_a") == "chosen" and entry.get("style_b") == "rejected":


        processed_data.append({
            "prompt": entry["prompt"],
            "chosen": entry["completion_a"],
            "rejected": entry["completion_b"]
        })
    else:
        print(f"Warning: Skipping an entry due to unexpected style fields: {entry.get('style_a')}, {entry.get('style_b')}")
        continue

print(f"Successfully processed {len(processed_data)} entries into DPO format.")

# 3. Convert the list of dictionaries to a Hugging Face Dataset

if processed_data:
    data_dict = {
        "prompt": [item["prompt"] for item in processed_data],
        "chosen": [item["chosen"] for item in processed_data],
        "rejected": [item["rejected"] for item in processed_data]
    }
    hf_dataset = Dataset.from_dict(data_dict)
    print("\nCreated Hugging Face Dataset:")
    print(hf_dataset)

    # 4. Create train/validation splits
    # test_size=0.2 means 20% for the test (validation) set.

    train_test_split = hf_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
    dataset_splits = DatasetDict({
        'train': train_test_split['train'],
        'validation': train_test_split['test']
    })

    print("\nDataset splits created:")
    print(dataset_splits)
    print(f"\nTraining examples: {len(dataset_splits['train'])}")
    print(f"Validation examples: {len(dataset_splits['validation'])}")


    print("\nExample from training set:")
    print(dataset_splits['train'][0])

    if len(dataset_splits['validation']) > 0:
        print("\nExample from validation set:")
        print(dataset_splits['validation'][0])
else:
    print("\nNo data processed, skipping Dataset creation and splitting.")


In [ ]:
# This cell was used to make sure that specified model paramters are trainable

# Check if model parameters are trainable
print("Checking trainable parameters status:")
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print(f"✓ Parameter {name} is trainable")



lora_model.train()  # Set to training mode explicitly

print(f"Model training mode: {lora_model.training}")

In [ ]:
# this cell is used to free up gpu memory before training since it will be needed

# Free up memory before training
import gc
import torch

# Clear cache
gc.collect()
torch.cuda.empty_cache()



## 4. Configure and Run DPO Training

This section sets up and executes DPOtraining process.

**Key steps include:**

- **Define Maximum Lengths:**
    - `MAX_PROMPT_LENGTH`: I seted the maximum length for input prompt to 128 tokens . This is based on the characteristics of the preference dataset.
    - `MAX_LENGTH`: I setted the maximum for the combined prompt and completion to 1200 tokens.
- **DPOConfig:** (justifying values of certain parameteres)
    - `num_train_epochs`: I setted number of epochs(iteration over the whole dataset) to 3. ( mainly because dataset is small and limited GPU resources)
    - `gradient_accumulation_steps`: I accumulated gradients over 4 steps before an optimizer update which somehow simulates increasing batch size (or training steps) while managing memory.
    - `learning_rate`:  Set to 1e-4 (common value I would say).
    - `bf16`: Enables bfloat16 mixed-precision training.
    - `logging_strategy` / `logging_steps`: log training metrics (every 5 steps here for evaluation and visualization purposes later).
    - `eval_strategy` / `eval_steps`: Evaluate the model on the validation set (after every 10 training steps).
- **Instantiate `DPOTrainer`:**
    - `model`: The LoRA-adapted model (`lora_model`) is provided as the model to be trained.
    - `args`: The `dpo_config` object containing all training parameters.
    - `train_dataset` / `eval_dataset`: The prepared training and validation dataset splits.
    - `processing_class` (`tokenizer`): The tokenizer used for processing the data.
    - `peft_config`: The LoRA configuration (`lora_config`) used for the model.
- **Start Training:** `dpo_trainer.train()` initiates the training process.


In [ ]:
from trl import DPOTrainer, DPOConfig

# I chosed these values in this way based on the preference dataset
MAX_PROMPT_LENGTH = 128
MAX_LENGTH = 1200

# Configure DPO training
dpo_config = DPOConfig(
    output_dir="./dpo_results_qwen_1.5b",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    bf16=True,
    logging_strategy="steps",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    report_to="tensorboard",
    remove_unused_columns=False,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_length=MAX_LENGTH,
)

# Instantiate the DPOTrainer
dpo_trainer = DPOTrainer(
    model=lora_model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset_splits['train'],
    eval_dataset=dataset_splits['validation'],
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("DPOTrainer initialized with Qwen2 1.5B model. Ready for training.")

dpo_trainer.train()


dpo_trainer.save_model("./dpo_results_qwen_1.5b/final_checkpoint")
tokenizer.save_pretrained("./dpo_results_qwen_1.5b/final_checkpoint")


## 5. Evaluate Model Performance and Visualize Evaluation Metrics

This section is dedicated to evaluating the trained DPO model and visualizing its performance over time (training steps)

**Key steps include:**

- **Define Visualization Function:** The `plot_training_metrics` function generates plots for:
    - Validation loss over training steps
    - Reward scores for chosen and rejected completions
    - Preference accuracy (how often the model prefers the correct completion) in my case the accuracy was actually 100 percent after only 10 training steps. I think part of it is also due to the power of the model (with 1.5 B parameters) and probably because the validation dataset is small(20 entries)
    - Reward margins (difference between chosen and rejected rewards). the margin always increased over the whole training process.
    - Log probabilities for chosen and rejected completions
    - Model logits for chosen and rejected completions
- **Evaluate the Model:** Use the DPO trainer’s `.evaluate()` method to compute evaluation metrics on the validation set.
- **Extract and Display Metrics:** Print summary statistics for log probabilities and preference accuracy.
- **Visualize Training Progress:** Call `plot_training_metrics` to display the evolution of metrics during training.

These visualizations and metrics provide insights into how well the model has learned to prefer the desired style (preference accuracy is 100 percent) and can help identify areas for further improvement.


 > **Note:** The metrics plots were generated during the initial run, but due to the free GPU runtime ending, my runime was suddenly disconnected. that is why they are not displayed in this notebook. If you would like to see the plots and metrics, please re-run this cell.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm


# Visualize evaluatuion metrics over time
def plot_training_metrics(trainer):


    logs_df = pd.DataFrame(trainer.state.log_history)
    print(f"Available metrics: {logs_df.columns.tolist()}")
     # this ensures that plots values are sorted by step
    logs_df = logs_df.sort_values('step')


    fig, axes = plt.subplots(3, 2, figsize=(18, 15))
    fig.suptitle('DPO Training Metrics', fontsize=16)

    # 1. Validation Loss
    ax = axes[0, 0]
    if 'eval_loss' in logs_df.columns:
        ax.plot(logs_df['step'], logs_df['eval_loss'], 'b-o', label='Validation Loss')
        ax.set_title('DPO Validation Loss')
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Loss')
        ax.grid(True, alpha=0.3)

    # 2. Reward Scores
    ax = axes[0, 1]
    metrics = ['eval_rewards/chosen', 'eval_rewards/rejected']
    for metric in metrics:
        if metric in logs_df.columns:
            label = metric.split('/')[-1].capitalize()
            ax.plot(logs_df['step'], logs_df[metric], '-o', label=label)
    ax.set_title('DPO Reward Scores')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Reward')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. Accuracy
    ax = axes[1, 0]
    if 'eval_rewards/accuracies' in logs_df.columns:
        ax.plot(logs_df['step'], logs_df['eval_rewards/accuracies'], 'g-o', label='Preference Accuracy')
        ax.axhline(y=0.5, color='r', linestyle='--', label='Random Guessing')
        ax.set_title('Preference Accuracy')
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Accuracy')
        ax.set_ylim([0.4, 1.05])  # Set y-axis range
        ax.legend()
        ax.grid(True, alpha=0.3)

    # 4. Reward Margins
    ax = axes[1, 1]
    if 'eval_rewards/margins' in logs_df.columns:
        ax.plot(logs_df['step'], logs_df['eval_rewards/margins'], 'm-o', label='Reward Margin')
        ax.set_title('Preference Margin (Chosen - Rejected)')
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Margin')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # 5. Log Probabilities
    ax = axes[2, 0]
    metrics = ['eval_logps/chosen', 'eval_logps/rejected']
    for metric in metrics:
        if metric in logs_df.columns:
            label = metric.split('/')[-1].capitalize()
            ax.plot(logs_df['step'], logs_df[metric], '-o', label=label)
    ax.set_title('Log Probabilities')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Log Probability')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 6. Logits
    ax = axes[2, 1]
    metrics = ['eval_logits/chosen', 'eval_logits/rejected']
    for metric in metrics:
        if metric in logs_df.columns:
            label = metric.split('/')[-1].capitalize()
            ax.plot(logs_df['step'], logs_df[metric], '-o', label=label)
    ax.set_title('Model Logits')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Logit Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()








# Execute the evaluation
print("Starting model evaluation...")
eval_results = dpo_trainer.evaluate()


# Extract metrics
metrics = eval_results
logps_chosen = metrics.get("eval_logps/chosen", 0)
logps_rejected = metrics.get("eval_logps/rejected", 0)
accuracy = metrics.get("eval_rewards/accuracies", 0)

print(f"\nEvaluation Metrics:")
print(f"Logps Chosen (mean): {logps_chosen:.4f}")
print(f"Logps Rejected (mean): {logps_rejected:.4f}")
print(f"Preference Accuracy: {accuracy:.4f}")




plot_training_metrics(dpo_trainer)



## 6. Evaluate the Performance of the Base Model

This section evaluates the base model (Qwen/Qwen2-1.5B-Instruct) on the validation set to compare performance againt my fine-tuned model.
**Key steps include:**

- **Define Evaluation Function:** The `evaluate_base_model` function:
    - Loads the base model with quantization.
    - Iterates over the validation set, computing log probabilities for both the "chosen" and "rejected" completions for each prompt.
    - Determines whether the base model prefers the chosen completion and calculates preference accuracy.
    - Collects and prints summary statistics: preference accuracy, average log probabilities, and average margin between chosen and rejected completions.
    - Visualizes results with bar plots and a histogram of preference margins.




In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

def evaluate_base_model(model_id, tokenizer, validation_dataset, max_length=768):

    # Configure quantization for base model
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Set model to evaluation mode
    base_model.eval()

    results = []
    correct_count = 0

    # Process each validation example
    print("Evaluating base model on validation data...")
    for i, example in enumerate(tqdm(validation_dataset)):
        prompt = example["prompt"]
        chosen = example["chosen"]
        rejected = example["rejected"]


        chosen_text = prompt + tokenizer.eos_token + chosen
        rejected_text = prompt + tokenizer.eos_token + rejected

        with torch.no_grad():
            # Process chosen completion
            chosen_input = tokenizer(chosen_text, return_tensors="pt", truncation=True, max_length=max_length).to("cuda")
            chosen_output = base_model(**chosen_input)
            chosen_logp = chosen_output.logits.mean().float().item()

            # Process rejected completion
            rejected_input = tokenizer(rejected_text, return_tensors="pt", truncation=True, max_length=max_length).to("cuda")
            rejected_output = base_model(**rejected_input)
            rejected_logp = rejected_output.logits.mean().float().item()

            # Does the base model prefer the chosen completion?
            prefers_chosen = chosen_logp > rejected_logp
            if prefers_chosen:
                correct_count += 1

            # Save result
            results.append({
                "prompt": prompt[:50] + "..." if len(prompt) > 50 else prompt,
                "chosen_logp": chosen_logp,
                "rejected_logp": rejected_logp,
                "prefers_chosen": prefers_chosen,
                "margin": chosen_logp - rejected_logp
            })


    accuracy = correct_count / len(validation_dataset)
    avg_chosen_logp = np.mean([r["chosen_logp"] for r in results])
    avg_rejected_logp = np.mean([r["rejected_logp"] for r in results])
    margins = [r["margin"] for r in results]

    # Print results
    print("\n===== Base Model Results =====")
    print(f"Preference accuracy: {accuracy:.4f} ({correct_count}/{len(validation_dataset)})")
    print(f"Average chosen logP: {avg_chosen_logp:.4f}")
    print(f"Average rejected logP: {avg_rejected_logp:.4f}")
    print(f"Average margin: {np.mean(margins):.4f}")

    # Create visualizations
    plt.figure(figsize=(15, 10))

    # 1. Preference accuracy
    plt.subplot(2, 2, 1)
    plt.bar(['Incorrect', 'Correct'],
            [(1-accuracy)*100, accuracy*100],
            color=['red', 'green'])
    plt.title('Base Model Preference Accuracy')
    plt.ylabel('Percentage')
    plt.ylim(0, 100)
    plt.grid(axis='y', alpha=0.3)

    # Add percentage labels
    plt.text(0, (1-accuracy)*100 + 3, f"{(1-accuracy)*100:.1f}%", ha='center')
    plt.text(1, accuracy*100 + 3, f"{accuracy*100:.1f}%", ha='center')

    # 2. Log probability comparison
    plt.subplot(2, 2, 2)
    plt.bar(['Chosen', 'Rejected'], [avg_chosen_logp, avg_rejected_logp])
    plt.title('Average Log Probabilities')
    plt.ylabel('Log Probability')
    plt.grid(axis='y', alpha=0.3)

    # 3. Margin histogram
    plt.subplot(2, 2, 3)
    plt.hist(margins, bins=15, alpha=0.7)
    plt.axvline(x=0, color='r', linestyle='--', label='Decision Boundary')
    plt.title('Distribution of Preference Margins')
    plt.xlabel('Margin (Chosen - Rejected LogP)')
    plt.ylabel('Count')
    plt.legend()
    plt.grid(True, alpha=0.3)




# Run the evaluation on base model
base_model_results = evaluate_base_model(model_id, tokenizer, dataset_splits["validation"])